In [ ]:
import pandas as pd
from tabulate import tabulate
# ============================================
# SOURCE 1 — Transactions (Master table)
# ============================================
transactions=pd.DataFrame(
    [{"transaction_id": 12343, "customer_id": 101, "amount": 10000, "status": "Active"},
    {"transaction_id": 12344, "customer_id": 102, "amount": 19000, "status": "Active"},
    {"transaction_id": 12345, "customer_id": 103, "amount": 90000, "status": "Inactive"},
    {"transaction_id": 12346, "customer_id": 104, "amount": 75000, "status": "Active"},
    {"transaction_id": 12347, "customer_id": 105, "amount": 25000, "status": "Inactive"},
    {"transaction_id": 12348, "customer_id": 106, "amount": 15000, "status": "Active"},
    ]
    )
# ============================================
# SOURCE 2 — Customers (Lookup table)
# ============================================
customers = pd.DataFrame(
    [{"customer_id": 101, "customer_name": "SAN", "city": "New York",  "segment": "Premium"},
    {"customer_id": 102, "customer_name": "TAN", "city": "Chicago",   "segment": "Standard"},
    {"customer_id": 103, "customer_name": "UAN", "city": "Houston",   "segment": "Premium"},
    {"customer_id": 104, "customer_name": "VAN", "city": "New York",  "segment": "Premium"},
    {"customer_id": 105, "customer_name": "WAN", "city": "Seattle",   "segment": "Standard"},
    # NOTE: customer 106 is MISSING — to show outer join behavior
    ]
    )
##print("=== Transactions ===\n",transaction.to_string(index=False))
print("=== Transactions ===\n",tabulate(transactions,headers='keys',tablefmt='psql',showindex=False))

print("\n=== Customers ===\n",tabulate(customers,headers='keys',tablefmt='psql',showindex=False))


 Types of Joins — Just Like Informatica!

In [ ]:
# INNER JOIN — only matching rows (default)
# Like Informatica Joiner: Join Type = Normal
inner = pd.merge(transactions,customers, on ="customer_id",how="inner")
#print("=======INNER JOIN========")
print("=======INNER JOIN========\n",tabulate(inner,headers="keys",tablefmt="psql",showindex=False))

# LEFT JOIN — all transactions, matching customer info
# Like Informatica Joiner: Join Type = Master Outer
left=pd.merge(transactions,customers,on="customer_id",how="left")
print("========LEFT JOIN========\n",tabulate(left,headers="keys",tablefmt="psql",showindex=False))

# RIGHT JOIN — all customers, matching transactions
right=pd.merge(transactions,customers,on ="customer_id", how="right")
print("=========RIGHT JOIN=======\n",tabulate(right,headers="keys",tablefmt="psql",showindex=False))

## OUTER JOIN — everything from both sides
outer=pd.merge(transactions,customers,on="customer_id",how="outer")
print("=========OUTER JOIN======\n",tabulate(outer,headers="keys",tablefmt="psql",showindex=False))

Lesson 2 — groupby() — Aggregator Transformation

In [64]:
# Use the LEFT JOIN result for aggregations
df=left.copy()
print (tabulate(df,headers="keys",tablefmt="psql",showindex=False))
# ============================================
# TOTAL AMOUNT PER CITY
# Like: Group by CITY, Sum AMOUNT
# ============================================
#print(tabulate(df,headers="keys",tablefmt="psql",showindex=False))
# city_totals=df.groupby("city")["amount"].sum().reset_index()
# city_totals.columns=["city","total_amount"]
# print(city_totals.to_string(index=False))

# ============================================
# MULTIPLE AGGREGATIONS AT ONCE
# Like: Group by SEGMENT, get Count + Sum + Avg
# ============================================
segment_status = df.groupby("segment").agg(
transaction_count = ("transaction_id" , "count"),
total_amount=("amount","sum"),
avg_amount=("amount","mean"),
max_amount=("amount","max"),
min_amount=("amount","min")
).reset_index()
print(segment_status.to_string(index=False))

# ============================================
# GROUP BY MULTIPLE COLUMNS
# Like: Group by CITY + STATUS
# ============================================
city_status=df.groupby(["city","status"])["amount"].sum().reset_index()
print(city_status.to_string(index=False))



+------------------+---------------+----------+----------+-----------------+----------+-----------+
|   transaction_id |   customer_id |   amount | status   | customer_name   | city     | segment   |
|------------------+---------------+----------+----------+-----------------+----------+-----------|
|            12343 |           101 |    10000 | Active   | SAN             | New York | Premium   |
|            12344 |           102 |    19000 | Active   | TAN             | Chicago  | Standard  |
|            12345 |           103 |    90000 | Inactive | UAN             | Houston  | Premium   |
|            12346 |           104 |    75000 | Active   | VAN             | New York | Premium   |
|            12347 |           105 |    25000 | Inactive | WAN             | Seattle  | Standard  |
|            12348 |           106 |    15000 | Active   | nan             | nan      | nan       |
+------------------+---------------+----------+----------+-----------------+----------+-----------+
